In [10]:
import pandas as pd
import json

In [11]:
file_path = '../data/miband/20260406_8278074507_MiFitness_ams1_data_copy/20260406_8278074507_MiFitness_hlth_center_aggregated_fitness_data.csv'
df_raw = pd.read_csv(file_path)

In [14]:
df_sleep = df_raw[(df_raw['Key'] == 'sleep') & (df_raw['Tag'] == 'daily_report')].copy()

# Parse JSON values and normalize into separate columns
sleep_data_list = [json.loads(val) for val in df_sleep['Value']]
df_sleep_normalized = pd.json_normalize(sleep_data_list)

# Unpack segment_details (list of dicts) - explode and normalize
if 'segment_details' in df_sleep_normalized.columns:
    df_sleep_exploded = df_sleep_normalized.explode('segment_details')
    segment_details_normalized = pd.json_normalize(df_sleep_exploded['segment_details'])
    df_sleep_exploded = df_sleep_exploded.drop('segment_details', axis=1).reset_index(drop=True)
    segment_details_normalized = segment_details_normalized.reset_index(drop=True)

    # Drop duplicate columns from segment_details_normalized (keep from df_sleep_exploded)
    duplicate_cols = segment_details_normalized.columns.intersection(df_sleep_exploded.columns)
    segment_details_normalized = segment_details_normalized.drop(columns=duplicate_cols)

    df_sleep_normalized = df_sleep_exploded.join(segment_details_normalized)

# Combine with original metadata (Time, Uid, etc.)
df_sleep_result = df_sleep[['Uid', 'Sid', 'Time']].reset_index(drop=True).join(df_sleep_normalized)

# Drop specified columns
columns_to_drop = ['total_turn_over', 'sleep_trace_duration', 'sleep_manually_duration', 'total_snore',
                   'breath_quality', 'total_body_move', 'total_snore_disturb', 'sleep_algorithm_version']
df_sleep_result = df_sleep_result.drop(columns=[col for col in columns_to_drop if col in df_sleep_result.columns])

df_sleep_result


,Uid,Sid,Time,total_duration,sleep_rem_duration,sleep_nap_duration,avg_hr,avg_spo2,sleep_light_duration,total_long_duration,...,awake_count,max_hr,sleep_deep_duration,min_hr,min_spo2,max_spo2,wake_up_time,duration,timezone,bedtime
0,8278074507,default,1736899200,428,125,0,56,0,167,428,...,2,80,136,48,NaN,NaN,1736930100,428,4,1736902380
1,8278074507,default,1736985600,515,119,65,54,0,226,450,...,1,79,105,43,NaN,NaN,1737012900,450,4,1736985060
2,8278074507,default,1737072000,515,119,65,54,0,226,450,...,1,79,105,43,NaN,NaN,1737022140,65,4,1737018240
3,8278074507,default,1737158400,496,121,0,57,0,247,496,...,2,99,128,47,NaN,NaN,1737100800,496,4,1737070020
4,8278074507,default,1737244800,523,121,0,50,0,246,523,...,0,79,156,43,NaN,NaN,1737185400,523,4,1737154020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,8278074507,default,1775088000,691,140,53,66,97,294,638,...,1,85,204,51,94.0,99.0,1769070600,638,4,1769032140
222,8278074507,default,1775174400,691,140,53,66,97,294,638,...,1,85,204,51,94.0,99.0,1769116620,53,4,1769113440
223,8278074507,default,1775260800,832,285,0,62,97,337,832,...,4,94,210,51,94.0,99.0,1769165760,832,4,1769113440
224,8278074507,default,1775347200,644,136,0,54,97,316,644,...,3,82,192,43,93.0,99.0,1769331360,644,4,1769292060


In [16]:
# Convert Time columns from epoch to datetime
df_sleep_result['Time'] = pd.to_datetime(df_sleep_result['Time'], unit='s')

# Convert wake_up_time and bedtime to datetime, then extract time in HH:MM:SS format
df_sleep_result['wake_up_time'] = pd.to_datetime(df_sleep_result['wake_up_time'], unit='s').dt.strftime('%H:%M:%S')
df_sleep_result['bedtime'] = pd.to_datetime(df_sleep_result['bedtime'], unit='s').dt.strftime('%H:%M:%S')

df_sleep_result


,Uid,Sid,Time,total_duration,sleep_rem_duration,sleep_nap_duration,avg_hr,avg_spo2,sleep_light_duration,total_long_duration,...,awake_count,max_hr,sleep_deep_duration,min_hr,min_spo2,max_spo2,wake_up_time,duration,timezone,bedtime
0,8278074507,default,2025-01-15,428,125,0,56,0,167,428,...,2,80,136,48,NaN,NaN,08:35:00,428,4,00:53:00
1,8278074507,default,2025-01-16,515,119,65,54,0,226,450,...,1,79,105,43,NaN,NaN,07:35:00,450,4,23:51:00
2,8278074507,default,2025-01-17,515,119,65,54,0,226,450,...,1,79,105,43,NaN,NaN,10:09:00,65,4,09:04:00
3,8278074507,default,2025-01-18,496,121,0,57,0,247,496,...,2,99,128,47,NaN,NaN,08:00:00,496,4,23:27:00
4,8278074507,default,2025-01-19,523,121,0,50,0,246,523,...,0,79,156,43,NaN,NaN,07:30:00,523,4,22:47:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,8278074507,default,2026-04-02,691,140,53,66,97,294,638,...,1,85,204,51,94.0,99.0,08:30:00,638,4,21:49:00
222,8278074507,default,2026-04-03,691,140,53,66,97,294,638,...,1,85,204,51,94.0,99.0,21:17:00,53,4,20:24:00
223,8278074507,default,2026-04-04,832,285,0,62,97,337,832,...,4,94,210,51,94.0,99.0,10:56:00,832,4,20:24:00
224,8278074507,default,2026-04-05,644,136,0,54,97,316,644,...,3,82,192,43,93.0,99.0,08:56:00,644,4,22:01:00


In [22]:
import os

output_dir = '../data/processed_data'
output_path = os.path.join(output_dir, 'miband_sleep_processed.csv')
df_sleep_result.to_csv(output_path, index=False)